# Vancouver Crime Type Classifier

## Overview
This project applies K-Nearest Neighbours (KNN) classification to predict 
crime type based on historical incident data from the Vancouver Police 
Department. Model performance is evaluated on a held-out test set to ensure 
generalisation to unseen data.

## Data Source
- **Provider:** Vancouver Police Department (VPD) GeoDASH Open Data
- **Dataset:** Crime data from 2003 to present, updated weekly
- **URL:** https://geodash.vpd.ca/opendata/

## Features Used
| Feature | Description |
|---|---|
| `NEIGHBOURHOOD` | Vancouver neighbourhood where incident occurred |
| `HOUR` | Hour of day (0-23) |
| `MONTH` | Month of year (1-12) |
| `DAY_OF_WEEK` | Day of week derived from date |
| `IS_WEEKEND` | Whether incident occurred on a weekend |
| `SEASON` | Season derived from month |
| `YEAR` | Year of incident |

## Target Variable
`TYPE` — Crime category (e.g. Theft, Break & Enter, Assault, Mischief)

## Methodology
1. Data loading and exploration
2. Data cleaning and feature engineering
3. Train/test split
4. KNN model training
5. Model evaluation and results

## Tools & Libraries
- **Language:** R
- **Environment:** JupyterLab
- **Key packages:** tidyverse, class, caret

In [2]:
install.packages(c(
  "tidyverse",   
  "caret",   
  "class",       
  "lubridate"    
))


The downloaded binary packages are in
	/var/folders/xy/cz8fxbvn7mv8c_d1fxb7sl5r0000gn/T//RtmpE8Jbep/downloaded_packages


In [3]:
library(tidyverse)
library(caret)
library(class)
library(lubridate)

Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘lubridate’ was built under R version 4.4.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.2     ✔ tibble    3.2.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: lattice


Attaching package: ‘caret’


The following object is masked from ‘package:purrr’:

    lift




In [6]:
crimes <- read_csv("/Users/kathyzhao/Crime-Classifier/crimedata_csv_AllNeighbourhoods_AllYears.csv")

Rows: 913777 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): TYPE, HUNDRED_BLOCK, NEIGHBOURHOOD
dbl (7): YEAR, MONTH, DAY, HOUR, MINUTE, X, Y

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [15]:
head(crimes)

TYPE,YEAR,MONTH,DAY,HOUR,MINUTE,HUNDRED_BLOCK,NEIGHBOURHOOD,X,Y,DATE,DAY_OF_WEEK
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<date>,<chr>
Break and Enter Commercial,2023,9,14,3,30,10XX ALBERNI ST,West End,491065.3,5459130,2023-09-14,Thursday
Break and Enter Commercial,2024,2,24,4,8,10XX BARCLAY ST,West End,490865.2,5458841,2024-02-24,Saturday
Break and Enter Commercial,2023,4,1,4,7,10XX BEACH AVE,West End,490197.9,5458239,2023-04-01,Saturday
Break and Enter Commercial,2025,4,28,12,5,10XX BEACH AVE,West End,490227.2,5458210,2025-04-28,Monday
Break and Enter Commercial,2023,5,11,18,0,10XX BEACH AVE,Central Business District,490249.2,5458167,2023-05-11,Thursday
Break and Enter Commercial,2024,2,11,22,5,10XX BEACH AVE,Central Business District,490249.2,5458167,2024-02-11,Sunday


## Data Preparation

The raw dataset contains separate YEAR, MONTH, and DAY columns. These are 
combined into a full date to extract DAY_OF_WEEK, since the same day number 
falls on different weekdays across years. DAY_OF_WEEK is a stronger predictor 
of crime type than the raw day number alone.

In [24]:
crimes <- crimes |> 
    mutate(DATE = as_date(paste(YEAR, MONTH, DAY, sep = "-")),
          DAY_OF_WEEK = weekdays(DATE),
          TYPE = as_factor(TYPE),
          NEIGHBOURHOOD = as_factor(NEIGHBOURHOOD),
          DAY_OF_WEEK = as_factor(DAY_OF_WEEK)) |>
    select(TYPE, YEAR, MONTH, DAY, HOUR, NEIGHBOURHOOD, DATE, DAY_OF_WEEK)
head(crimes)

TYPE,YEAR,MONTH,DAY,HOUR,NEIGHBOURHOOD,DATE,DAY_OF_WEEK
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<date>,<fct>
Break and Enter Commercial,2023,9,14,3,West End,2023-09-14,Thursday
Break and Enter Commercial,2024,2,24,4,West End,2024-02-24,Saturday
Break and Enter Commercial,2023,4,1,4,West End,2023-04-01,Saturday
Break and Enter Commercial,2025,4,28,12,West End,2025-04-28,Monday
Break and Enter Commercial,2023,5,11,18,Central Business District,2023-05-11,Thursday
Break and Enter Commercial,2024,2,11,22,Central Business District,2024-02-11,Sunday


In [25]:
crimes_clean <- crimes |>
    select(TYPE, YEAR, MONTH, HOUR, NEIGHBOURHOOD, DAY_OF_WEEK)
head(crimes_clean)

TYPE,YEAR,MONTH,HOUR,NEIGHBOURHOOD,DAY_OF_WEEK
<fct>,<dbl>,<dbl>,<dbl>,<fct>,<fct>
Break and Enter Commercial,2023,9,3,West End,Thursday
Break and Enter Commercial,2024,2,4,West End,Saturday
Break and Enter Commercial,2023,4,4,West End,Saturday
Break and Enter Commercial,2025,4,12,West End,Monday
Break and Enter Commercial,2023,5,18,Central Business District,Thursday
Break and Enter Commercial,2024,2,22,Central Business District,Sunday


## Encoding Categorical Variables

KNN calculates distance between data points using numeric values, so categorical 
variables must be converted to numeric values. NEIGHBOURHOOD and 
DAY_OF_WEEK are encoded as numeric factors, where each unique category is 
assigned an integer. These are then scaled alongside the other numeric 
predictors.

In [31]:
crimes_numeric <- crimes_clean |>
    mutate(NEIGHBOURHOOD = as.numeric(NEIGHBOURHOOD),
          DAY_OF_WEEK = as.numeric(DAY_OF_WEEK)) |>
    drop_na()
head(crimes_numeric)

TYPE,YEAR,MONTH,HOUR,NEIGHBOURHOOD,DAY_OF_WEEK
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Break and Enter Commercial,2023,9,3,1,1
Break and Enter Commercial,2024,2,4,1,2
Break and Enter Commercial,2023,4,4,1,2
Break and Enter Commercial,2025,4,12,1,3
Break and Enter Commercial,2023,5,18,2,1
Break and Enter Commercial,2024,2,22,2,4


## Variable Reference

### Neighbourhood Encoding
| Number | Neighbourhood |
|---|---|
| 1 | West End |
| 2 | Central Business District |
| 3 | Strathcona |
| 4 | Grandview-Woodland |
| 5 | Sunset |
| 6 | Kensington-Cedar Cottage |
| 7 | Stanley Park |
| 8 | Shaughnessy |
| 9 | Victoria-Fraserview |
| 10 | Fairview |
| 11 | Hastings-Sunrise |
| 12 | Marpole |
| 13 | Oakridge |
| 14 | Kitsilano |
| 15 | Mount Pleasant |
| 16 | Riley Park |
| 17 | Kerrisdale |
| 18 | West Point Grey |
| 19 | Renfrew-Collingwood |
| 20 | Arbutus Ridge |
| 21 | Killarney |
| 22 | South Cambie |
| 23 | Dunbar-Southlands |
| 24 | Musqueam |

### Day of Week Encoding
| Number | Day |
|---|---|
| 1 | Thursday |
| 2 | Saturday |
| 3 | Monday |
| 4 | Sunday |
| 5 | Wednesday |
| 6 | Tuesday |
| 7 | Friday |

## Train/Test Split

The dataset is split into a training set (75%) and a test set (25%). 
Stratification by TYPE ensures that the proportion of each crime category 
is preserved in both sets. This prevents any crime type from being 
over or underrepresented.

In [47]:
install.packages("rsample")
library(rsample)
install.packages("recipes")
library(recipes)
install.packages("parsnip")
library(parsnip)


The downloaded binary packages are in
	/var/folders/xy/cz8fxbvn7mv8c_d1fxb7sl5r0000gn/T//RtmpE8Jbep/downloaded_packages

The downloaded binary packages are in
	/var/folders/xy/cz8fxbvn7mv8c_d1fxb7sl5r0000gn/T//RtmpE8Jbep/downloaded_packages

The downloaded binary packages are in
	/var/folders/xy/cz8fxbvn7mv8c_d1fxb7sl5r0000gn/T//RtmpE8Jbep/downloaded_packages


Warning message:
“package ‘parsnip’ was built under R version 4.4.3”


In [48]:
crimes_split <- initial_split(crimes_numeric, prop = 0.75, strata = TYPE)
crimes_training <- training(crimes_split)
crimes_testing <- testing(crimes_split)

## Recipe

A recipe is created to define the model and preprocessing steps. 
TYPE is specified as the target variable, with NEIGHBOURHOOD, HOUR, MONTH, 
YEAR, and DAY_OF_WEEK as predictors. All numeric predictors are centred and 
scaled.

In [49]:
crimes_recipe <- recipe(TYPE ~ YEAR	+ MONTH	+ HOUR + NEIGHBOURHOOD + DAY_OF_WEEK, data = crimes_training) |>
    step_scale(all_predictors()) |>
    step_center(all_predictors())

## Cross-Validation

The training data is split into 5 folds, stratified by TYPE, to tune the 
optimal value of K during model training.

In [50]:
crimes_vfold <- vfold_cv(crimes_training, v = 5, strata = TYPE)

## Model Specification

Specifies a KNN classifier using the kknn engine, with the number of 
neighbours K left to be tuned during cross-validation

In [51]:
crimes_spec <- nearest_neighbor(weight_func = "rectangular", neighbors = tune()) |>
    set_engine("kknn") |>
    set_mode("classification")